Team AD05: Orlando Scheil, Jameson Gibbs, Shritan Gopu, Owen Longano


## EE 242 Lab 1a – Modifying Signals - Sampling and Amplifying

In this lab, you will work as a team through a series of exercises to introduce you to working with audio signals and explore the impact of different amplitude and time operations on signals. This is part a of two-week lab. You should plan on completing part a in the first week.

**Submission Format:** PDF

Please refer to Lab0 for how to convert your notebook to a PDF.

### Import Cell

We'll refer to the below cell as the "import cell." Every module you import should be imported in it. We've added some for you to start with.

In [ ]:
import numpy as np
import matplotlib
import scipy.signal as sig
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
import IPython.display as ipd

import numpy.typing as npt
# import whatever other modules you use in this lab -- there are more that you need than we've included

## Lab 1a.1 -- Working with sound files

Some starter code is given below.

In [ ]:
def timescale(x: npt.ArrayLike, fs: float, scale_factor: float):
    """Timescale the signal `x` with the given scale factor.

    Parameters
    ----------
    x : ndarray
        Input signal

    fs : float
        The original sampling rate in (Hz)

    a : float
        Scaling factor to apply to the original sampling rate.
        This has to be a decimal value for as_integer_ratio to work.

    Returns
    -------
    y : ndarray
        The timescaled signal
    t : ndarray
        The time samples corresponding to timescaled signal
    """
    n, d = (np.float64(scale_factor)).as_integer_ratio()
    Ts = 1 / fs

    y = sig.resample_poly(x, d, n)
    t = np.arange(0, len(y), 1) * Ts
    return y, t

def linear_timescale(x: np.ndarray[tuple[int, ...], np.dtype[np.float64]], fs: float, scale_factor: float):
    """Linearly timescale the signal `x` with the given scale factor.

    Parameters
    ----------
    x : ndarray
        Input signal

    fs : float
        The original sampling rate in (Hz)

    scale_factor : float
        Scaling factor to apply to the original sampling rate.
        This has to be a decimal value for as_integer_ratio to work.

    Returns
    -------
    y : ndarray
        Linearly resampled signal
    t_resample : ndarray
        Time samples corresponding to rescaled signal
    """

    n, d = (np.float64(scale_factor)).as_integer_ratio()

    positions = np.arange(0, len(x), n / d)
    y = np.interp(positions, np.arange(len(x)), x)
    t_resample = np.arange(len(y)) / fs

    return y, t_resample

**A.**  First, download the train.wav sound file provided. 

Read in the file saving the audio vector and sampling frequency in variables `x1` and `fs1`, respectively. Print the sampling rate `fs1` (which should be 32kHz) and the shape of `x1`. The shape of `x1` will tell you the length and number of channels.

1. Read the train.wav file

In [ ]:
fs1, x1 = wav.read("train32.wav")


2. Print the sampling rate, data type, and shape of signal

In [ ]:
print("Sampling Rate:", fs1)
print("Data Type:", x1.dtype)
print("Shape:", x1.shape)

**B.** Create two new signals `x2` and `x3` by changing the sampling rate of the original `x1` to `fs2` and `fs3`, respectively using the values given below:

- $f_{s_2} = \frac{1}{2}f_{s_1}$ (16 kHz)

- $f_{s_3} = \frac{3}{2}f_{s_1}$  (48 kHz)

This will have the effect of making it so that:
- $x_2(t)=x_1(t/2)$ 
- $x_3(t)=x_1(1.5t)$

1. Use both `timescale` and `linear_timescale` functions above to resample the signal using `scale_factor` values of 0.5 and 1.5 to generate $x_2(t)$ and $x_3(t)$, respectively. 

Note: You may want to change the datatype that the signals are encoded on to a floating point format before feeding them to either `timescale` or `linear_timescale`. See [numpy.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) for more details. Data type names can be found [here](https://numpy.org/doc/stable/reference/arrays.scalars.html#sized-aliases).

In [ ]:
x1_float = x1.astype(np.float64)

x2_timescale, t2_timescale = timescale(x1_float, fs1, 0.5)
x3_timescale, t3_timescale = timescale(x1_float, fs1, 1.5)

x2_linear_timescale, t2_linear_timescale = linear_timescale(x1_float, fs1, 0.5)
x3_linear_timescale, t3_linear_timescale = linear_timescale(x1_float, fs1, 1.5)

2. Save the resampled signals to `.wav` files, labeling them as follows:
 - `timescale` resampled: $x_2$ (low) $\to$ `train_low_poly.wav`, $x_3$ (high) $\to$ `train_high_poly.wav`
 - `linear_timescale` resampled: $x_2$ (low) $\to$ `train_low_linear.wav`, $x_3$ (high) $\to$ `train_high_linear.wav`

 Note: Make sure you convert the data arrays back to the the original data type found in Part A before writing them using `wav.write()`. See [numpy.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) for more details. Data type names can be found [here](https://numpy.org/doc/stable/reference/arrays.scalars.html#sized-aliases).

In [ ]:
signals = [
    ("train_low_ts.wav", x2_timescale),
    ("train_high_ts.wav", x3_timescale),
    ("train_low_lts.wav", x2_linear_timescale),
    ("train_high_lts.wav", x3_linear_timescale)
]

for filename, signal in signals:
    wav.write(filename, fs1, signal.astype(x1.dtype))

**C.**  Read in the three different versions of the train sound file and play each one.   ($x_1(t), x_2(t), x_3(t)$, don't change the $f_s$ while playing)

In [ ]:
filenames = [filename[0] for filename in signals]
filenames.insert(0, "train32.wav")


for filename in filenames:
    sample_rate, audio = wav.read(filename)
    ipd.display(ipd.Audio(audio.T, rate=fs1))

###  Discussion

Comment on how the audio changes when the incorrect sampling frequency is used.

---

Playing the audio add a lower rate makes the audio slower and lower pitched, while playing it at a higher rate makes the audio faster and higher pitched. Each altered signal is shaped by the factor that we gave it (0.5 and 1.5), which is shown across both timescale methods, however we had slight crackling in the timescale version. Our best guess as to why this happens is because resample_poly(...) filters the audio, but when we convert the audio back to the original dtype, it converts some peaks to integer which causes that crackling noise.

---

## Assignment 2 -- Amplitude Operations on Signals

**A.1.** Create a discrete time signal $s_1(t)$ that is the same length as $x_1(t)$ such that:

- $s_1(t) = 1$ for $t \in [0,0.5]$ 
- $s_1(t) = 0.2$ for $t > 0.5$.

Tips:

- Create a time array `t1` containing the time values corresponding to `x1`.

- NumPy allows logical slicing: `s1[t1 < 0.5]` selects all elements of `s1` where the condition `t1 < 0.5` is True.

- [`numpy.where()`](https://numpy.org/doc/stable/reference/generated/numpy.where.html) may be used instead of logical slicing if you prefer.

- Use [`numpy.zeros_like()`](https://numpy.org/doc/stable/reference/generated/numpy.zeros_like.html#numpy-zeros-like) to create an array of zeros with the same shape as `x1`, then assign values based on the condition above. Make sure to specify floating point as the datatype since `numpy.zeros_like()` also copies the datatype of `x1` by default.

In [ ]:
t1 = np.arange(len(x1)) / fs1
s1 = np.ones_like(x1, dtype=float)
s1[t1 > 0.5] = 0.2

**A.2.** Multiply `x1` with `s1` to create `v1`. Save this signal to a `.wav` file called `v1.wav`. 

In [ ]:
v1 = x1 * s1
wav.write("v1.wav", fs1, np.int16(v1))


**B.1** Create a discrete-time decaying ramp signal `r1`, that is the same length as `x1`, with values decreasing linearly from `r1[0] = 1` and `r1[-1] = 0`.

Tips:
- Use `numpy.linspace()`.

In [ ]:
r1 = np.linspace(1.0, 0.0, len(x1))

**B.2** Multiply x1 with r1 to create v2. Save this signal to a wav file.

Note: Make sure to convert v2 to the same data type as `x1` you found in part A.

In [ ]:
v2 = x1.astype(np.float64) * r1
wav.write("v2.wav", fs1, np.int16(v2))

**C.**  Read in `v1` and `v2` and play the two different modifications together with the original, to verify that the volume of the second whistle is reduced. 

Ensure to play all the signals for the TAs

In [ ]:
for filename in ["train32.wav", "v1.wav", "v2.wav"]:
    rate, signal = wav.read(filename)
    print(filename, "-", rate, "Hz")
    ipd.display(ipd.Audio(filename=filename))

**D.** Make two plots: one containing both v1 and x1, and another containing both v2 and x1, to compare the effects of the scaling.

Tips:

Plots are layered so that subsequent plots will paint over previous plots. You should plot `x1` so that it does not cover `v1` and `v2`, since `x1` is always greater than or equal to `v1` and `v2`.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(t1, x1, label="Original Signal (x1)")
plt.plot(t1, v1, label="v1")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Signals")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(t1, x1, label="Original Signal (x1)")
plt.plot(t1, v2, label="v2")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Signals")
plt.legend()
plt.grid(True)
plt.show()

###  Discussion

1. Discuss the differences that the two modifications have on the signal. 

The first modification (v1) has a steep dropoff in volume that stays constant for the rest of the sample, whereas the second one (v2) has a gradual decrease in volume.

2. What would happen if you defined `s1` to take value 2 for the $[0,0.5]$ range instead?

The base amplitude would be twice as big, thus making the samples louder

3. If you wanted a smooth but faster decay than linear in amplitude, what kinds of signals might you use? There are many possible answers for this question but it's sufficient to just list a couple.

you could have an exponential signal like a quadratic or cubic equation. This would make for a faster decay of the signal.